In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F


# Question 1: Install Spark and PySpark

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("hw6") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .getOrCreate()

In [3]:
spark

In [4]:
print(spark.version)

3.5.3


# Question 2: Yellow November 2025

In [5]:
df=spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [6]:
print(f"Total records: {df.count()}")

Total records: 4181444


In [7]:
df.repartition(4).write.parquet("yellow_2025_11_partitioned", mode="overwrite")

In [8]:
# Check file sizes
import os
files = [f for f in os.listdir("yellow_2025_11_partitioned") if f.endswith(".parquet")]
sizes = [os.path.getsize(f"yellow_2025_11_partitioned/{f}") / (1024*1024) for f in files]
print(f"Files: {files}")
print(f"Sizes (MB): {[round(s,2) for s in sizes]}")
print(f"Average size (MB): {round(sum(sizes)/len(sizes), 2)}")

Files: ['part-00000-6ac616d3-208a-4af4-bf6f-166ee743edcb-c000.snappy.parquet', 'part-00001-6ac616d3-208a-4af4-bf6f-166ee743edcb-c000.snappy.parquet', 'part-00002-6ac616d3-208a-4af4-bf6f-166ee743edcb-c000.snappy.parquet', 'part-00003-6ac616d3-208a-4af4-bf6f-166ee743edcb-c000.snappy.parquet']
Sizes (MB): [26.32, 26.34, 26.34, 26.35]
Average size (MB): 26.34


In [11]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [12]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

# Question 3 : Count Records

In [15]:
count_nov15 = df.filter(
    (F.to_date(F.col("tpep_pickup_datetime")) == "2025-11-15")
).count()
print(f"Trip on Nov 15: {count_nov15}")

Trip on Nov 15: 162604


# Question 4: Longest trip

In [21]:
df_duration = df.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) /3600
)
longest = df_duration.agg(F.max("duration_hours")).collect()[0][0]

print(f"Longest trip (hours): {round(longest, 1)}")

Longest trip (hours): 90.6


# Question 5: User Interface 

In [22]:
print("Spark UI runs on port: 4040")
print(f"Check it at: http://localhost:4040")

Spark UI runs on port: 4040
Check it at: http://localhost:4040


# Question 6: Least frequent pickup location zone

In [35]:
zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("taxi_zone_lookup.csv") 


In [36]:
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [43]:
# Join trip with zones on PULocationID
df_zones=df.join(zones, df.PULocationID == zones.LocationID, "left")
# Count pickups per zone, find least frequent
least_frequent=df_zones.groupby("Zone") \
    .count() \
    .orderBy("count") \
    .first()


In [48]:
print(f"Least frequent zone: {least_frequent['Zone']} ({least_frequent['count']} trips)") 

Least frequent zone: Arden Heights (1 trips)
